# Mini-TP 7 · Seguridad con SAIF — Aprendizaje federado

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Aprendizaje federado**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **clientes maliciosos** que envenenan el modelo global y **fuga de datos** por los updates.  
**Controles:** agregación robusta (mediana/trimmed) · *clipping* de la norma · privacidad diferencial.  
**Elementos SAIF:** detectar y actuar · mejora continua.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.


### Dependencias
```bash
uv add scikit-learn numpy
```

## 1. FedAvg y un cliente MALICIOSO
Repartimos los datos en 5 clientes (como en la Sesión 5). Un cliente malicioso envía un
update **escalado e invertido** (model poisoning). Con el promedio simple de FedAvg, ese
único cliente **arruina el modelo global**.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

d = load_digits(); X = d.data / 16.0; y = d.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
rng = np.random.RandomState(0)

def softmax(Z): Z = Z - Z.max(1, keepdims=True); e = np.exp(Z); return e / e.sum(1, keepdims=True)
def onehot(y, k=10): o = np.zeros((len(y), k)); o[np.arange(len(y)), y] = 1; return o
def init(): return rng.normal(0, .01, (64, 10)), np.zeros(10)
def entrenar_local(W, b, Xc, yc, ep=20, lr=0.5):
    W, b = W.copy(), b.copy()
    for _ in range(ep):
        P = softmax(Xc @ W + b); G = (P - onehot(yc)) / len(Xc)
        W -= lr * (Xc.T @ G); b -= lr * G.sum(0)
    return W, b
def acc(W, b): return (softmax(Xte @ W + b).argmax(1) == yte).mean()

idx = rng.permutation(len(Xtr)); parts = np.array_split(idx, 5)
clientes = [(Xtr[p], ytr[p]) for p in parts]

def federar(agg="media", atacar=False, R=15):
    W, b = init()
    for _ in range(R):
        Ws, bs = [], []
        for i, (Xc, yc) in enumerate(clientes):
            Wl, bl = entrenar_local(W, b, Xc, yc)
            if atacar and i == 0:                 # cliente 0 = malicioso
                Wl = W - 10 * (Wl - W); bl = b - 10 * (bl - b)
            Ws.append(Wl); bs.append(bl)
        Ws, bs = np.array(Ws), np.array(bs)
        if agg == "media":
            W, b = Ws.mean(0), bs.mean(0)
        else:                                      # agregación robusta
            W, b = np.median(Ws, 0), np.median(bs, 0)
    return acc(W, b)

print(f"sin ataque · FedAvg (media)      = {federar('media', False):.3f}")
print(f"con ataque · FedAvg (media)      = {federar('media', True):.3f}   <- colapsa")

## 2. Control principal (GUIADO) — agregación robusta (mediana por coordenada)

In [ ]:
print(f"con ataque · robusto (mediana)   = {federar('mediana', True):.3f}   <- resiste")
print("\nLa mediana por coordenada ignora los valores extremos del cliente malicioso.")
print("Es una defensa de 'detectar y actuar': tolera clientes que no son de confianza.")

## 3. Tu entrega — completa los `# TODO`

In [ ]:
# TODO 1: agrega CLIPPING de la norma del update de cada cliente antes de agregar
#   (si ||Wl - W|| > S, reescálalo a norma S). Sirve contra updates gigantes.
# TODO 2: agrega PRIVACIDAD DIFERENCIAL: ruido gaussiano al update agregado
#   (evita que los updates filtren datos de los clientes).
# TODO 3: prueba 'media recortada' (trimmed mean) en vez de mediana y compara.
# TODO 4: corre todo sobre TU modelo/dataset repartido en clientes.
print("Completa los TODO sobre tu propio modelo federado.")

## 4. Reflexión
1. ¿Por qué el promedio simple es tan frágil ante un solo cliente malicioso?
2. ¿Qué protege el *clipping* y qué protege la privacidad diferencial (son cosas distintas)?
3. ¿Qué elemento de SAIF cubre tu control principal?

*(Escribe tu respuesta aquí.)*